In [1]:
from langgraph.func import entrypoint
from langgraph.store.postgres import AsyncPostgresStore
from langchain_core.runnables import RunnableConfig
from langgraph.store.memory import InMemoryStore
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langmem import create_memory_store_manager, ReflectionExecutor

/home/frank_shan/miniconda3/envs/pyapi/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = OllamaEmbeddings(model="qwen3-embedding:8b")
chat_model = ChatOllama(model="qwen3:8b", temperature=0)

dummy_vec = embedding_model.embed_query("test")
EMBEDDING_DIM = len(dummy_vec)
print(f"Detected embedding dimension: {EMBEDDING_DIM}")

Detected embedding dimension: 4096


In [6]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
store = InMemoryStore(
    index={
        "dims": 4096,
        "embed": embedding_model,
    }
)

memory_manager = create_memory_store_manager(
    chat_model,
    store=store,
    namespace=("memories", "{user_id}"),
)

@entrypoint(store=store)
async def my_agent(message: str, config: RunnableConfig):
    memories = await memory_manager.asearch(
        query=message,
        config=config,
    )
    llm_response = chat_model.invoke([HumanMessage(content=message), SystemMessage(content=f"## Memories from the user:\n{str(memories)}")])
    response = {"role": "assistant", "content": llm_response.content}

    await memory_manager.ainvoke(
        {"messages": [{"role": "user", "content": message}, response]},
    )
    return response

config = {"configurable": {"user_id": "user123"}}
response_1 = await my_agent.ainvoke(
    "I prefer dark mode in all my apps",
    config=config,
    stream_mode="values"
)
print("response_1:", response_1)
# Later conversation - automatically retrieves and uses the stored preference
response_2 = await my_agent.ainvoke(
    "What theme do I prefer?",
    config=config,
    stream_mode="values"
)
print("response_2:", response_2)
# You can list over memories in the user's namespace manually:
print(memory_manager.search(query="app preferences", config=config))


response_1: {'role': 'assistant', 'content': "I'm glad to hear you prefer dark mode—it's a great choice for reducing eye strain and enhancing readability in low-light environments! 🌙 If you're looking for tips on enabling dark mode across apps, troubleshooting issues, or recommendations for apps that support it, feel free to let me know. I'd be happy to help!"}
response_2: {'role': 'assistant', 'content': 'You prefer **dark mode** across all applications. This preference was explicitly stated with confidence (p=1.0), and the benefits (e.g., reduced eye strain, improved readability in low light) were acknowledged. If you need help enabling or troubleshooting dark mode, feel free to ask! 🌑'}
[Item(namespace=['memories', 'user123'], key='1bca8f51-1d46-4da1-9535-9abe76b1e7c4', value={'content': 'User explicitly prefers dark mode across all applications (p=1.0). Agent acknowledged preference, affirmed its benefits (eye strain reduction, low-light readability), and offered assistance with en